# 2. One-click training and evaluation (Google Colab)

**How to use:** upload `waste-project.zip` to Colab, unzip it, open this notebook, then `Runtime → Run all`. If you already cloned your GitHub repository, simply `!git clone <your-private-repo-url>` instead of uploading.

Enable a GPU: `Runtime → Change runtime type → T4 GPU`. The cell below runs `run_all.py`, which downloads the dataset, creates the split, trains E1–E5, selects the best model on validation accuracy, fits temperature scaling on validation logits, evaluates once on the untouched test set, and writes all figures and `experiments/results/summary.json`.

In [ ]:
# OPTION A: clone your private repository (replace the URL)
# !git clone https://github.com/<your-org>/<your-repo>.git waste-project

# OPTION B: upload a zip of this project folder from your computer
# from google.colab import files
# uploaded = files.upload()
# import zipfile, glob
# zipfile.ZipFile(glob.glob('*.zip')[0]).extractall('.')

import os
print('Files here:', sorted(os.listdir('.'))[:20])

In [ ]:
import sys, subprocess
from pathlib import Path

project = Path('waste-project') if Path('waste-project/run_all.py').exists() else Path('.')
assert (project/'run_all.py').exists(), 'Put this notebook inside the waste-project folder first.'
subprocess.run([sys.executable, str(project/'run_all.py'),
                '--epochs-cnn', '8', '--epochs-mt', '8', '--epochs-finetune', '4'],
               cwd=project, check=True)

In [ ]:
import json
from pathlib import Path
summary = json.loads((Path('experiments/results/summary.json')).read_text())
print('Selected model:', summary['selected_model'])
print('Test accuracy:', summary['test']['temperature_scaled']['accuracy'])
print('ECE before/after:', summary['test']['ece_before'], summary['test']['ece_after'])
print('Abstention:', summary['test']['abstention_on_test'])

from IPython.display import Image, display
for name in ['curves_E3.png','confusion_matrix_final.png','calibration.png',
             'risk_coverage.png','gradcam_examples.png']:
    display(Image(filename=f'experiments/figures/{name}'))